<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0-B Initialized Fixed Grid Backtest
Fixed arithmetic spot grid. Initial BTC is sized so each seeded SELL grid holds exactly the BTC a normal BUY at that grid level would create. Trading logic is isolated in Section 1 for later Shadow/Live reuse.

Rules: SELL before BUY; BUY only on downward crossing; no same-candle sell/rebuy; same-candle SELL proceeds cannot fund BUYs; no compounding. This notebook does **not** send Binance orders.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import base64,bisect,heapq,json,os
import numpy as np
import pandas as pd
import requests

# 1. Trading System

## 1.1 Trading Configuration

In [ ]:
SYMBOL="BTCUSDT"
INITIAL_CAPITAL=3000.0
GRID_FLOOR=38000.0
GRID_CEILING=127000.0
GRID_GAP=1000.0
BUY_FEE=0.001
SELL_FEE=0.001

## 1.2 Grid, Initialization & Trading Engine

In [ ]:
def validate_config(capital,floor,ceiling,gap,bf,sf):
    if not(capital>0 and floor>0 and ceiling>floor and gap>0): raise ValueError("Invalid trading configuration.")
    if not(0<=bf<1 and 0<=sf<1): raise ValueError("Fees must be in [0,1).")
    n=(ceiling-floor)/gap
    if not np.isclose(n,round(n)): raise ValueError("Grid range must be exactly divisible by gap.")
    return int(round(n))

def build_grid(floor,ceiling,gap):
    b=np.arange(floor,ceiling,gap,dtype=float)
    return pd.DataFrame({'grid_id':np.arange(1,len(b)+1),'buy_price':b,'sell_target':b+gap})

def derive_grid(grid,start,capital,bf):
    g=grid.copy()
    if not(float(g.buy_price.min())<start<float(g.sell_target.max())): raise ValueError("Start price must be inside grid range.")
    seed=g.sell_target>start; reserve=~seed
    weight=int(reserve.sum())+float(np.sum(start/g.loc[seed,'buy_price'].to_numpy(float)))
    q=capital/weight
    g['order_size_usdt']=q; g['seed_at_start']=seed
    g['normal_net_btc']=q/g.buy_price*(1-bf)
    g['initial_entry_cost_usdt']=np.where(seed,g.normal_net_btc/(1-bf)*start,0.0)
    reserved=float(reserve.sum()*q); seeded=float(g.initial_entry_cost_usdt.sum())
    if not np.isclose(reserved+seeded,capital,atol=1e-8): raise AssertionError("Initial allocation mismatch.")
    return g,{'normal_order_size_usdt':float(q),'funding_weight':float(weight),'initial_sell_positions':int(seed.sum()),'initial_buy_levels':int(reserve.sum()),'reserved_cash_usdt':reserved,'seeded_btc_cost_usdt':seeded}

def make_position(tid,gi,time,entry,grid_buy,target,cost,pv,bf,sf,kind):
    gross=cost/entry if kind=='GRID_BUY' else (cost/grid_buy)/(1-bf)
    # For INITIAL_SEED, cost argument is normal order size; actual market cost is recalculated.
    if kind=='INITIAL_SEED':
        net_target=cost/grid_buy*(1-bf); gross=net_target/(1-bf); actual_cost=gross*entry
    else: actual_cost=cost
    fee_btc=gross*bf; btc=gross-fee_btc; gross_sell=btc*target; sell_fee=gross_sell*sf
    return {'trade_id':int(tid),'grid_index':int(gi),'entry_type':kind,'status':'OPEN','buy_time':time,'buy_price':float(entry),'sell_target':float(target),'sell_time':pd.NaT,'order_size_usdt':float(actual_cost),'portfolio_value_at_buy':float(pv),'order_pct_of_portfolio':float(actual_cost/pv),'portfolio_value_at_sell':np.nan,'btc_amount':float(btc),'buy_fee_btc':float(fee_btc),'sell_fee_usdt':float(sell_fee),'net_sell_usdt':float(gross_sell-sell_fee),'net_pnl':np.nan}

def event(eid,time,side,tid,gi,price,cash_move,grid_cf,cb,ca,bb,ba,pv,init=False):
    return {'event_id':eid,'time':time,'side':side,'trade_id':tid,'grid_index':gi,'price':price,'cash_movement':cash_move,'grid_cashflow':grid_cf,'cash_before':cb,'cash_after':ca,'btc_before':bb,'btc_after':ba,'portfolio_value_before':pv,'initialization_trade':init}

def initialize(data,template,capital,bf,sf):
    time=data.open_time.iloc[0]; start=float(data.open.iloc[0]); g,s=derive_grid(template,start,capital,bf)
    pos={}; active={}; heap=[]; ev=[]; cash=float(capital); btc=fee=0.0; tid=eid=0
    for gi in g.index[g.seed_at_start]:
        r=g.loc[gi]; pv=cash+btc*start; tid+=1
        p=make_position(tid,gi,time,start,float(r.buy_price),float(r.sell_target),float(r.order_size_usdt),pv,bf,sf,'INITIAL_SEED')
        cb,bb=cash,btc; cash-=p['order_size_usdt']; btc+=p['btc_amount']; fee+=p['buy_fee_btc']*start
        pos[tid]=p; active[gi]=tid; heapq.heappush(heap,(p['sell_target'],tid)); eid+=1
        ev.append(event(eid,time,'BUY',tid,gi,start,-p['order_size_usdt'],0,cb,cash,bb,btc,pv,True))
    init={**s,'start_time':time,'start_price':start,'initial_cash':float(cash),'initial_btc':float(btc),'initial_btc_cost_usdt':s['seeded_btc_cost_usdt'],'initial_buy_fee_usdt':float(fee),'initial_btc_allocation_pct':float(s['seeded_btc_cost_usdt']/capital*100)}
    return g,pos,active,heap,ev,cash,btc,tid,eid,init,fee

def run_grid(data,template,capital,bf,sf):
    data=data.sort_values('open_time').reset_index(drop=True).copy()
    g,pos,active,heap,ev,cash,btc,tid,eid,init,buy_fees=initialize(data,template,capital,bf,sf)
    q=float(init['normal_order_size_usdt']); bp=g.buy_price.to_numpy(float); st=g.sell_target.to_numpy(float); bl=bp.tolist()
    eq=np.empty(len(data)); cv=np.empty(len(data)); bv=np.empty(len(data)); sell_fees=realized=0.0; completed=0; prev=None; tol=1e-12
    for i,c in enumerate(data.itertuples(index=False)):
        t,o,h,l,cl=c.open_time,float(c.open),float(c.high),float(c.low),float(c.close); cash_start=cash; sold=set()
        while heap and heap[0][0]<=h+tol:
            _,x=heapq.heappop(heap); p=pos[x]
            if p['status']!='OPEN': continue
            gi=p['grid_index']; cb,bb=cash,btc; pv=cb+bb*p['sell_target']; cash+=p['net_sell_usdt']; btc-=p['btc_amount']
            if abs(btc)<1e-12: btc=0.0
            pnl=p['net_sell_usdt']-p['order_size_usdt']; p.update(status='CLOSED',sell_time=t,portfolio_value_at_sell=pv,net_pnl=float(pnl)); active.pop(gi,None)
            sell_fees+=p['sell_fee_usdt']; realized+=pnl; completed+=1; sold.add(gi); eid+=1
            ev.append(event(eid,t,'SELL',x,gi,p['sell_target'],p['net_sell_usdt'],pnl,cb,cash,bb,btc,pv))
        budget=cash_start; top=o if prev is None else max(prev,o)
        if l<top:
            a=bisect.bisect_left(bl,l); z=bisect.bisect_left(bl,top)
            for gi in range(z-1,a-1,-1):
                if gi in active or gi in sold: continue
                if budget+tol<q: break
                buy,target=float(bp[gi]),float(st[gi]); cb,bb=cash,btc; pv=cb+bb*buy; tid+=1
                p=make_position(tid,gi,t,buy,buy,target,q,pv,bf,sf,'GRID_BUY'); budget-=q; cash-=q; btc+=p['btc_amount']; buy_fees+=p['buy_fee_btc']*buy
                pos[tid]=p; active[gi]=tid; heapq.heappush(heap,(target,tid)); eid+=1
                ev.append(event(eid,t,'BUY',tid,gi,buy,-q,0,cb,cash,bb,btc,pv))
        eq[i],cv[i],bv[i]=cash+btc*cl,cash,btc; prev=cl
    curve=pd.DataFrame({'open_time':data.open_time,'close':data.close,'cash':cv,'btc':bv,'equity':eq})
    return {'grid':g,'initialization':init,'positions':pos,'trade_event_log':pd.DataFrame(ev),'equity_curve':curve,'final_cash':float(cash),'final_btc':float(btc),'realized_profit':float(realized),'completed_cycles':int(completed),'total_buy_fee_usdt':float(buy_fees),'total_sell_fee_usdt':float(sell_fees)}

NUMBER_OF_GRIDS=validate_config(INITIAL_CAPITAL,GRID_FLOOR,GRID_CEILING,GRID_GAP,BUY_FEE,SELL_FEE)
GRID_TEMPLATE=build_grid(GRID_FLOOR,GRID_CEILING,GRID_GAP)
print(f"Number of Grids : {NUMBER_OF_GRIDS}")

# 2. Backtest System

## 2.1 Configuration, Data & Metrics

In [ ]:
TIMEFRAME="1m"; START_DATE="2024-01-01"; END_DATE="2026-01-01"
DATA_DIR="/content/drive/MyDrive/03.Trading/00.Live Trading"

def load_data():
    p=os.path.join(DATA_DIR,f"{SYMBOL}-{TIMEFRAME}-combined.csv"); d=pd.read_csv(p); d.open_time=pd.to_datetime(d.open_time,utc=True)
    for c in ['open','high','low','close','volume']: d[c]=d[c].astype(float)
    a,b=pd.Timestamp(START_DATE,tz='UTC'),pd.Timestamp(END_DATE,tz='UTC')
    d=d.drop_duplicates('open_time').sort_values('open_time').loc[lambda x:(x.open_time>=a)&(x.open_time<b)].reset_index(drop=True)
    if d.empty: raise ValueError("No data in selected period.")
    return d

def perf(d,curve,capital):
    e=curve.equity.to_numpy(float); peak=np.maximum.accumulate(e); dd=e/peak-1; final=float(e[-1]); ret=final/capital-1
    days=(d.open_time.iloc[-1]-d.open_time.iloc[0]).total_seconds()/86400; ann=np.expm1(np.log(final/capital)*365.25/days) if days>0 and final>0 else np.nan
    cal=float(ann/abs(dd.min())) if dd.min()<0 and np.isfinite(ann) else np.nan; curve=curve.copy(); curve['drawdown']=dd
    return {'final_equity':final,'net_return':float(ret),'annualized_return':float(ann),'max_drawdown':float(dd.min()),'calmar_ratio':cal,'equity_curve':curve}

def buy_hold(d,capital,bf):
    entry=float(d.open.iloc[0]); gross=capital/entry; net=gross*(1-bf); curve=pd.DataFrame({'open_time':d.open_time,'close':d.close,'cash':0.0,'btc':net,'equity':net*d.close})
    s=perf(d,curve,capital); return {'entry_price':entry,'net_btc':float(net),'entry_fee_usdt':float(capital*bf),**s}

## 2.2 Trade History, Audit & Results

In [ ]:
def history(pos,final_time,final_price):
    r=[]
    for tid in sorted(pos):
        p=pos[tid]; closed=p['status']=='CLOSED'; sell=p['sell_time'] if closed else pd.NaT; pv=p['portfolio_value_at_sell'] if closed else np.nan
        pnl=p['net_pnl'] if closed else p['btc_amount']*final_price-p['order_size_usdt']; hold=(sell if closed else final_time)-p['buy_time']
        r.append({'Trade ID':p['trade_id'],'Status':p['status'],'Buy Time':p['buy_time'],'Buy Price':p['buy_price'],'Sell Target':p['sell_target'],'Sell Time':sell,'Order Size (USDT)':p['order_size_usdt'],'Portfolio Value at Buy':p['portfolio_value_at_buy'],'Order % of Portfolio':p['order_pct_of_portfolio']*100,'Portfolio Value at Sell':pv,'Net P&L':float(pnl),'Holding Time':hold})
    return pd.DataFrame(r)

def audit(res,hist,capital,bf):
    init=res['initialization']; ev=res['trade_event_log']; curve=res['equity_curve']; q=init['normal_order_size_usdt']; seed_ok=True
    for p in res['positions'].values():
        if p['entry_type']=='INITIAL_SEED':
            exp=q/float(res['grid'].loc[p['grid_index'],'buy_price'])*(1-bf)
            if not np.isclose(p['btc_amount'],exp,atol=1e-12,rtol=0): seed_ok=False; break
    open_btc=sum(p['btc_amount'] for p in res['positions'].values() if p['status']=='OPEN')
    return {'cash_reconciliation':np.isclose(capital+ev.cash_movement.sum(),res['final_cash'],atol=1e-8),'realized_profit_reconciliation':np.isclose(ev.grid_cashflow.sum(),res['realized_profit'],atol=1e-8),'equity_identity':np.max(np.abs(curve.cash+curve.btc*curve.close-curve.equity))<=1e-8,'cash_never_negative':curve.cash.min()>=-1e-8,'initial_allocation_reconciliation':np.isclose(init['initial_cash']+init['initial_btc_cost_usdt'],capital,atol=1e-8),'all_grid_slots_initialized_or_reserved':init['initial_sell_positions']+init['initial_buy_levels']==len(res['grid']),'initial_seed_quantity_matches_normal_grid':seed_ok,'final_btc_matches_open_positions':np.isclose(res['final_btc'],open_btc,atol=1e-10),'closed_trade_count_reconciliation':hist.Status.eq('CLOSED').sum()==res['completed_cycles']}

df_1m=load_data(); result=run_grid(df_1m,GRID_TEMPLATE,INITIAL_CAPITAL,BUY_FEE,SELL_FEE); stats=perf(df_1m,result['equity_curve'],INITIAL_CAPITAL); result['equity_curve']=stats['equity_curve']; bh=buy_hold(df_1m,INITIAL_CAPITAL,BUY_FEE)
trade_history=history(result['positions'],df_1m.open_time.iloc[-1],float(df_1m.close.iloc[-1])); open_n=int(trade_history.Status.eq('OPEN').sum()); unrl=float(trade_history.loc[trade_history.Status.eq('OPEN'),'Net P&L'].sum())
summary={'initial_capital':INITIAL_CAPITAL,'initial_market_price':result['initialization']['start_price'],'normal_order_size_usdt':result['initialization']['normal_order_size_usdt'],'initial_cash':result['initialization']['initial_cash'],'initial_btc':result['initialization']['initial_btc'],'initial_sell_positions':result['initialization']['initial_sell_positions'],'initial_buy_levels':result['initialization']['initial_buy_levels'],'final_equity':stats['final_equity'],'net_return':stats['net_return'],'annualized_return':stats['annualized_return'],'max_drawdown':stats['max_drawdown'],'calmar_ratio':stats['calmar_ratio'],'completed_cycles':result['completed_cycles'],'open_positions':open_n,'final_cash':result['final_cash'],'final_btc':result['final_btc'],'realized_profit':result['realized_profit'],'unrealized_pnl':unrl,'total_fee_usdt_equiv':result['total_buy_fee_usdt']+result['total_sell_fee_usdt']}
benchmark_summary={k:bh[k] for k in ['entry_price','net_btc','entry_fee_usdt','final_equity','net_return','annualized_return','max_drawdown','calmar_ratio']}; comparison_summary={'excess_return_vs_buy_hold':stats['net_return']-bh['net_return'],'drawdown_improvement_vs_buy_hold':stats['max_drawdown']-bh['max_drawdown']}
comparison_table=pd.DataFrame({'Metric':['Final Equity (USDT)','Net Return','Annualized Return','Max Drawdown','Calmar Ratio'],'V0-B Grid':[f"{stats['final_equity']:,.2f}",f"{stats['net_return']:.2%}",f"{stats['annualized_return']:.2%}",f"{stats['max_drawdown']:.2%}",f"{stats['calmar_ratio']:.3f}"],'BTC Buy & Hold':[f"{bh['final_equity']:,.2f}",f"{bh['net_return']:.2%}",f"{bh['annualized_return']:.2%}",f"{bh['max_drawdown']:.2%}",f"{bh['calmar_ratio']:.3f}"]})
audit_checks=audit(result,trade_history,INITIAL_CAPITAL,BUY_FEE); AUDIT_STATUS='PASS' if all(bool(v) for v in audit_checks.values()) else 'FAIL'
print('===== V0-B RESULT ====='); [print(f"{k:32s}: {v}") for k,v in summary.items()]; print('\n===== V0-B vs BTC BUY & HOLD ====='); display(comparison_table); print(f"Excess Return vs Buy & Hold       : {comparison_summary['excess_return_vs_buy_hold']:.2%}"); print(f"Drawdown Improvement vs Buy & Hold: {comparison_summary['drawdown_improvement_vs_buy_hold']:.2%}"); print('\nAUDIT:',AUDIT_STATUS)
if AUDIT_STATUS!='PASS': raise AssertionError('V0-B AUDIT FAILED')

## 2.3 User Trade History

In [ ]:
display(trade_history)

# 3. Logging System

In [ ]:
REPO="natdanaiii/Trading"; BRANCH="main"; GITHUB_SUMMARY="logs/latest_v0_backtest_log.json"; GITHUB_HISTORY="logs/latest_v0_trade_history.csv"; LOCAL_SUMMARY="/content/latest_v0_backtest_log.json"; LOCAL_HISTORY="/content/latest_v0_trade_history.csv"
def safe(v):
    if v is pd.NaT or v is pd.NA:return None
    if isinstance(v,dict):return {str(k):safe(x) for k,x in v.items()}
    if isinstance(v,(list,tuple,np.ndarray)):return [safe(x) for x in list(v)]
    if isinstance(v,(bool,np.bool_)):return bool(v)
    if isinstance(v,(int,np.integer)):return int(v)
    if isinstance(v,(float,np.floating)):return None if not np.isfinite(v) else float(v)
    if isinstance(v,pd.Timestamp):return v.isoformat()
    return v
payload=safe({'log_schema_version':5,'strategy':'V0-B Initialized Fixed Grid (Grid-Consistent Seed Sizing)','run_info':{'generated_at_utc':pd.Timestamp.now(tz='UTC').isoformat(),'repository':REPO,'branch':BRANCH,'notebook':'Grid_trading_V0.ipynb','symbol':SYMBOL,'timeframe':TIMEFRAME,'start_date':START_DATE,'end_date':END_DATE,'data_rows':len(df_1m),'data_first_time':df_1m.open_time.min(),'data_last_time':df_1m.open_time.max()},'trading_config':{'initial_capital':INITIAL_CAPITAL,'floor':GRID_FLOOR,'ceiling':GRID_CEILING,'gap':GRID_GAP,'buy_fee':BUY_FEE,'sell_fee':SELL_FEE},'initialization':result['initialization'],'summary':summary,'buy_hold_benchmark':benchmark_summary,'comparison_vs_buy_hold':comparison_summary,'audit':{'status':AUDIT_STATUS,'checks':audit_checks},'trade_history_file':GITHUB_HISTORY})
with open(LOCAL_SUMMARY,'w') as f:json.dump(payload,f,indent=2,allow_nan=False)
trade_history.to_csv(LOCAL_HISTORY,index=False)
try:
    from google.colab import userdata
    token=userdata.get('GITHUB_TOKEN')
except: token=None
def upload(path,local,msg):
    url=f'https://api.github.com/repos/{REPO}/contents/{path}'; h={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json'}; old=requests.get(url,headers=h,timeout=30); b={'message':msg,'content':base64.b64encode(open(local,'rb').read()).decode(),'branch':BRANCH}
    if old.status_code==200:b['sha']=old.json()['sha']
    r=requests.put(url,headers=h,json=b,timeout=30);r.raise_for_status();return r.json()['commit']['sha']
if token:
    print('Summary Commit:',upload(GITHUB_SUMMARY,LOCAL_SUMMARY,'Update latest V0-B backtest log'));print('History Commit:',upload(GITHUB_HISTORY,LOCAL_HISTORY,'Update latest V0-B trade history'))
else:print("GitHub upload SKIPPED: Colab Secret 'GITHUB_TOKEN' not found.")